# Profile a table, then test it on every change

<a href="https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/11_profile_and_test_a_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**For:** anyone asked to "create test cases for our production tables" — data governance, analysts,
data engineers. No warehouse needed: the table lives in DuckDB inside this notebook.

Generic test templates are either too simple ("check for nulls") or too convoluted, because they
don't know your data. This notebook does it the other way round:

1. **Profile** the table — what is actually in it — and let LakeLogic point at what looks wrong.
2. **Agree the rules** with whoever owns the table, and write them down as a **data contract**.
3. **Run the rules** on every load and every change: good rows pass, bad rows are set aside with
   the reason.

The table is a health-insurance **claims** table with the problems real loads have.

In [ ]:
!pip install -q "lakelogic[polars,duckdb]"
import lakelogic
import duckdb
import pandas as pd

print("lakelogic", lakelogic.__version__)

## The table

5,000 claims. Hidden in them: a repeated claim id, a service date in the future, a negative
amount, a status nobody uses (`pending_review`) and an empty diagnosis code. Nobody has told
LakeLogic any of this.

In [ ]:
import random
from datetime import date, timedelta

random.seed(7)
con = duckdb.connect()
members = [f"M{i:05d}" for i in range(1, 801)]
rows = []
for i in range(1, 5001):
    service = date(2026, 1, 1) + timedelta(days=random.randint(0, 270))
    paid = service + timedelta(days=random.randint(1, 40)) if random.random() > 0.3 else None
    rows.append(
        [
            f"C{i:06d}",
            random.choice(members),
            service,
            paid,
            round(random.uniform(20, 4000), 2),
            random.choice(["submitted", "approved", "denied", "paid"]),
            random.choice(["E11.9", "I10", "J45.909", "M54.5", "Z00.00"]),
        ]
    )

rows[10][0] = rows[9][0]  # a repeated claim id
rows[20][2] = date(2027, 3, 1)  # a service date in the future
rows[30][4] = -125.00  # a negative amount
rows[40][5] = "pending_review"  # a status nobody uses
rows[60][6] = None  # an empty diagnosis code

con.execute("""create table claims (claim_id varchar, member_id varchar, service_date date,
               paid_date date, amount decimal(10,2), status varchar, diagnosis_code varchar)""")
con.executemany("insert into claims values (?, ?, ?, ?, ?, ?, ?)", rows)
con.sql("select * from claims limit 5").df()

## Step 1 — Profile it

`assess_table` reads the table and returns two things: a **profile** (per column: type, how much
is empty, how many distinct values, the range) and **findings** — what a person should look at.
Only counts and summaries come back, never rows. Point it at your own table by passing a Spark
session, a DuckDB connection or a SQLAlchemy engine instead of `con`.

In [ ]:
from lakelogic.core.profile_assess import assess_table

entry = assess_table("claims", con)

profile = pd.DataFrame(
    [
        {
            "column": c["name"],
            "type": c["type"],
            "% empty": c.get("null_pct"),
            "distinct": c.get("distinct_exact"),
            "min": c.get("min"),
            "max": c.get("max"),
        }
        for c in entry["profile"]["columns"]
    ]
)
profile

In [ ]:
def show(findings):
    return pd.DataFrame([{"kind": f["kind"], "finding": f["title"], "evidence": f.get("evidence")} for f in findings])


print(entry["finding_counts"])
show(entry["findings"])

**What it found on its own:** the repeated `claim_id` (it worked out that `claim_id` is the key),
that `paid_date` is empty on about 30% of rows, and that no contract governs the table.

Each finding has a kind:

* **confirmed** — a rule you agreed failed (none yet: we haven't agreed any).
* **potential** — an observation to check with the owner. Is an empty `paid_date` fine? Yes, for
  unpaid claims. Is a repeated `claim_id` fine? No.
* **gap** — a check that is missing.

Read the profile too: `service_date` runs to **2027** and `amount` goes **below zero**. Those are
not errors to LakeLogic yet, because nobody has said what is allowed. That is step 2.

## Step 2 — Agree the rules, write them down

Sit down with the table's owner, go through the profile, and write what you agree as a **data
contract**: one YAML file per table, kept in version control next to the table's code. Every rule
here comes from the profile above, not from a template.

In [ ]:
CONTRACT = """
version: 1.0.0
dataset: claims
info:
  title: Claims
  owner: claims-data@example.org
  domain: claims
  target_layer: silver
primary_key: [claim_id]             # profile: claim_id is the key
model:
  fields:
    - {name: claim_id, type: string, required: true}
    - {name: member_id, type: string, required: true}
    - {name: service_date, type: date, required: true}
    - {name: paid_date, type: date}  # empty is allowed: the claim is not paid yet
    - {name: amount, type: double, required: true}   # double: 64-bit, right for money
    - name: status
      type: string
      required: true
      accepted_values: [submitted, approved, denied, paid]
    - {name: diagnosis_code, type: string, required: true}
quality:
  row_rules:
    - accepted_values: {name: status_is_known, field: status, values: [submitted, approved, denied, paid]}
    - name: amount_not_negative
      sql: "amount >= 0"
    - name: service_date_not_in_future
      sql: "service_date <= current_date"
    - name: paid_after_service
      sql: "paid_date IS NULL OR paid_date >= service_date"
"""
open("claims.yaml", "w").write(CONTRACT)

# The same profile, now checked against what you agreed.
entry = assess_table("claims", con, contract="claims.yaml")
print(entry["finding_counts"])
show(entry["findings"])

With the contract, the repeated key, the unknown status and the empty diagnosis code are now
**confirmed** issues — rules you agreed, failing. The remaining **gap** says what the contract
still lacks (for example, how fresh the data must be).

## Step 3 — Run the rules on every load and every change

The contract is also what runs. Every row is checked: good rows pass, bad rows are set aside
(**quarantined**) with the rule they broke — nothing is silently dropped. Run this in your
pipeline on every load, and in CI on every pull request that changes the table or its contract.
That is "whenever we make a change, we test again".

The repeated `claim_id` is not in the list below: a duplicate is a fact about the *table*, not one
row, so the profile in step 2 reports it. Run both: the rules on every load, the profile on a
schedule.

In [ ]:
df = con.sql("select * from claims").pl()
good, bad = lakelogic.DataProcessor("claims.yaml", engine="polars").run(df)
print(f"{len(df):,} rows in · {len(good):,} passed · {len(bad):,} set aside")

bad.select(["claim_id", "service_date", "status", "diagnosis_code", "_lakelogic_errors"]).to_pandas()

## Check the notebook did what it says

In [ ]:
first = assess_table("claims", con)
titles = {f["title"] for f in first["findings"]}
assert any("claim_id" in t for t in titles), titles
assert any("paid_date" in t for t in titles), titles

agreed = assess_table("claims", con, contract="claims.yaml")
assert agreed["finding_counts"]["confirmed"] >= 3, agreed["finding_counts"]

caught = {r["claim_id"]: " ".join(r["_lakelogic_errors"]) for r in bad.to_dicts()}
assert "service_date_not_in_future" in caught["C000021"]
assert "amount_not_negative" in caught["C000031"]
assert "status_is_known" in caught["C000041"]
assert "diagnosis_code" in caught["C000061"]
assert len(good) + len(bad) == len(df)  # every row accounted for
print("All checks passed.")

## Use it on your own tables

* **Profile:** `assess_table("catalog.schema.table", spark)` on Databricks, or pass a DuckDB
  connection or SQLAlchemy engine. Add `settings={"mode": "sample", "sample_unit": "percent",
  "sample_percent": 10}` for a large table.
* **Many tables at once:** `assess_tables([...], connection)` from the same module.
* **Agree and keep the rules:** one contract YAML per table, in version control.
* **Test on every change:** run the contract in CI on each pull request, and on a schedule against
  production.